In [78]:
import cv2
import mediapipe as mp
import numpy as np

from mediapipe.tasks import python
from mediapipe.tasks.python import vision

MARGIN = 10  # pixels
FONT_SIZE = 1
FONT_THICKNESS = 1
HANDEDNESS_TEXT_COLOR = (88, 205, 54)  # vibrant green
MAX_HANDS = 2
CAMERA = 0
FPS = 30
REC_TEST = False
frame_idx = 0

In [79]:
def create_hand_detector(model_path: str = "hand_landmarker.task", num_hands: int = 1):
    """
    MediaPipe HandLandmarker 생성.
    """
    base_options = python.BaseOptions(model_asset_path=model_path)
    options = vision.HandLandmarkerOptions(
        base_options=base_options,
        num_hands=num_hands,
        running_mode=mp.tasks.vision.RunningMode.VIDEO,
    )
    detector = vision.HandLandmarker.create_from_options(options)
    return detector

def get_drawing_utils():
    """
    랜드마크 시각화를 위한 연결 정보 반환.
    """
    return vision.HandLandmarksConnections.HAND_CONNECTIONS

def capture_frame(cap: cv2.VideoCapture):
    """
    웹캠에서 프레임 1장을 읽어 반환.
    실패 시 (False, None) 반환.
    """
    ret, frame = cap.read()
    frame = cv2.flip(frame, 1)
    if not ret:
        return False, None
    return True, frame

def convert_bgr_to_mp_image(bgr_frame: np.ndarray):
    """
    OpenCV BGR 이미지를 RGB 및 MediaPipe Image로 변환.
    """
    rgb_frame = cv2.cvtColor(bgr_frame, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)
    return rgb_frame, mp_image

def detect_hand_landmarks(detector, bgr_frame: np.ndarray):
    """
    BGR 프레임을 받아 MediaPipe HandLandmarker로 손 랜드마크 검출.
    반환:
      - rgb_frame
      - detection_result
    """
    global frame_idx
    rgb_frame, mp_image = convert_bgr_to_mp_image(bgr_frame)
    timestamp_ms = int(frame_idx * 1000 / FPS)
    # detection_result = detector.detect(mp_image)
    detection_result = detector.detect_for_video(mp_image, timestamp_ms)
    frame_idx += 1


    hand_landmarks_list = getattr(detection_result, "hand_landmarks", [])
    handedness_list = getattr(detection_result, "handedness", [])

    for idx in range(len(hand_landmarks_list)):
        hand_label = handedness_list[idx][0].category_name

        if hand_label == "Right":
            handedness_list[idx][0].category_name = "Left"
        elif hand_label == "Left":
            handedness_list[idx][0].category_name = "Right"

    return rgb_frame, detection_result

def draw_landmarks_on_image(rgb_image, detection_result, hand_connections):
    hand_landmarks_list = detection_result.hand_landmarks
    handedness_list = detection_result.handedness
    annotated_image = np.copy(rgb_image)
    image_h, image_w, _ = annotated_image.shape

    for idx, hand_landmarks in enumerate(hand_landmarks_list):
        # 관절 연결선 그리기
        for connection in hand_connections:
            start_idx, end_idx = connection.start, connection.end
            start_lm = hand_landmarks[start_idx]
            end_lm = hand_landmarks[end_idx]
            start_point = (int(start_lm.x * image_w), int(start_lm.y * image_h))
            end_point = (int(end_lm.x * image_w), int(end_lm.y * image_h))
            cv2.line(annotated_image, start_point, end_point, (0, 255, 255), 2)

        # 관절 포인트 그리기
        for lm in hand_landmarks:
            px = int(lm.x * image_w)
            py = int(lm.y * image_h)
            cv2.circle(annotated_image, (px, py), 4, (255, 80, 80), -1)

        if idx < len(handedness_list):
            handedness = handedness_list[idx]
            x_coordinates = [landmark.x for landmark in hand_landmarks]
            y_coordinates = [landmark.y for landmark in hand_landmarks]
            text_x = int(min(x_coordinates) * image_w)
            text_y = int(min(y_coordinates) * image_h) - MARGIN

            cv2.putText(
                annotated_image,
                f"{handedness[0].category_name}",
                (text_x, text_y),
                cv2.FONT_HERSHEY_DUPLEX,
                FONT_SIZE,
                HANDEDNESS_TEXT_COLOR,
                FONT_THICKNESS,
                cv2.LINE_AA,
            )
    return annotated_image

In [80]:
import pyautogui

prev_pos = None


def predict_hand_gesture(detection_result):
    global prev_pose

    hand_landmarks_list = getattr(detection_result, "hand_landmarks", [])
    handedness_list = getattr(detection_result, "handedness", [])

    if not hand_landmarks_list:
        return {"left": None, "right": None}

    for idx in range(len(hand_landmarks_list)):
        hand_landmarks = hand_landmarks_list[idx]
        hand_label = handedness_list[idx][0].category_name

        if hand_label != "Right":
            continue

        hans_side = 1
        joint = np.array([[lm.x, lm.y, lm.z] for lm in hand_landmarks])
        v = joint[[0, 5, 9, 13, 17]] - 0.5

        if prev_pos is None:
            prev_pose = v.mean(axis=0)
        pos = prev_pose * 0.9 + v.mean(axis=0) * 0.1

        print(pos)
        prev_pose = pos

        # curr_pointer = pyautogui.position

        pyautogui.move(pos[0] * 200, pos[1] * 200)

    prev_pose = None

In [81]:
cap = cv2.VideoCapture(CAMERA)
if not cap.isOpened():
    raise RuntimeError(f"웹캠을 열 수 없습니다. camera_index={CAMERA}")

detector = create_hand_detector(model_path="hand_landmarker.task", num_hands=MAX_HANDS)
hand_connections = get_drawing_utils()
# knn = load_knn()
try:
    while True:
        ret, frame = capture_frame(cap)
        if not ret:
            print("프레임을 읽지 못했습니다. 종료합니다.")
            break

        rgb_frame, detection_result = detect_hand_landmarks(detector, frame)
        annotated_image = draw_landmarks_on_image(rgb_frame, detection_result, hand_connections)

        predict_hand_gesture(detection_result)
        # TODO Modulization
        handedness_list = getattr(detection_result, "handedness", [])


        # OpenCV는 BGR 기준으로 표시하므로 RGB -> BGR로 변환
        display_image = cv2.cvtColor(annotated_image, cv2.COLOR_RGB2BGR)

        font = cv2.FONT_HERSHEY_SIMPLEX
        cv2.imshow("MediaPipe Hand Landmarks", display_image)

        # ESC 또는 q 입력 시 종료
        key = cv2.waitKey(1) & 0xFF
        if key == 27 or key == ord("q"):
            break
        
finally:
    cap.release()
    cv2.waitKey(-1)
    cv2.destroyAllWindows()

[ 0.40774351  0.30304216 -0.55983772]
[ 0.33213799  0.12328265 -0.50991306]
[ 0.33880652  0.06485696 -0.5602446 ]
[ 0.29572403  0.03774922 -0.54730987]
[ 0.28947619  0.0366532  -0.56246114]
[ 0.20005181  0.04014453 -0.55923327]
[ 0.12420231 -0.01518089 -0.54855622]
[ 0.06717533 -0.03133374 -0.54202456]
[-0.02877644  0.01722175 -0.5484918 ]
[-0.14123401  0.130068   -0.55287208]
[-0.27048542  0.18212628 -0.53219403]
[-0.3398306   0.24127303 -0.52279825]
[-0.38285532  0.26693276 -0.5396001 ]
[-0.36499391  0.29085754 -0.52160192]
[-0.34528312  0.18841155 -0.52055981]
[-0.24633661  0.12593749 -0.56898703]
[-0.1314983   0.05106459 -0.56340986]
[-0.04139032 -0.04994271 -0.55548817]
[-0.03149254 -0.0615925  -0.55933668]
[-0.03366268 -0.04758325 -0.56132191]
[-0.0393387  -0.05520681 -0.5685287 ]
[-0.04070914 -0.05672057 -0.55867338]
[-0.04149926 -0.05108399 -0.55760198]
[-0.0238649  -0.0593244  -0.55600197]
[ 0.00977144 -0.0844492  -0.55494349]
[ 0.03192428 -0.07329395 -0.55379348]
[ 0.06451061

In [82]:
detection_result

HandLandmarkerResult(handedness=[], hand_landmarks=[], hand_world_landmarks=[])

In [83]:
import pyautogui
pyautogui.position()

Point(x=643, y=460)